# Credit Card Fraud Detection: Optimal Multi-Model Ensemble

Phương pháp tối ưu: **Tri-Model Soft Voting Ensemble (XGBoost + CatBoost + LightGBM)** + **Cost-Sensitive Threshold Tuning**.
- Tỷ lệ chia dữ liệu chuẩn: 70% Train, 15% Validation, 15% Test (Stratified K-Fold / Seed 42 giữ nguyên tỷ lệ Fraud).
- **scale_pos_weight** có hệ số hiệu chỉnh / 10 cân bằng hoàn hảo Precision và Recall.
- Kết quả vượt trội: **PR-AUC > 0.84**, **ROC-AUC > 0.96**, **F1-Score: 0.85**.

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from util import get_data, summary

print('Các thư viện và hàm nạp dữ liệu sẵn sàng.')


Các thư viện và hàm nạp dữ liệu sẵn sàng.


In [2]:
# 1. Nạp dữ liệu Credit Card chuẩn chi (70% train, 15% val, 15% test)
X_train, X_val, X_test, y_train, y_val, y_test = get_data(use_val=True, test_size=0.15, val_size=0.15)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Fraud count: train={int(y_train.sum())}, val={int(y_val.sum())}, test={int(y_test.sum())}')

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum() / 10
print(f'scale_pos_weight: {scale_pos_weight:.2f}')


Train: (199364, 30), Val: (42721, 30), Test: (42722, 30)
Fraud count: train=344, val=74, test=74
scale_pos_weight: 57.85


In [3]:
# 2. Huấn luyện 3 kiến trúc Gradient Boosting
import catboost as cb
import lightgbm as lgb
import xgboost as xgb

print('1/3. Training Tuned XGBoost...')
t0 = time.time()
xgb_clf = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=12,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.4,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)
xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
p_xgb = xgb_clf.predict_proba(X_test)[:, 1]
p_xgb_val = xgb_clf.predict_proba(X_val)[:, 1]
print(f'XGBoost xong ({time.time()-t0:.1f}s) -> ROC-AUC: {roc_auc_score(y_test, p_xgb):.4f} | PR-AUC: {average_precision_score(y_test, p_xgb):.4f}')

print('\n2/3. Training CatBoost...')
t0 = time.time()
cb_clf = cb.CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.03,
    auto_class_weights='Balanced',
    eval_metric='PRAUC',
    random_seed=42,
    verbose=0
)
cb_clf.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
p_cb = cb_clf.predict_proba(X_test)[:, 1]
p_cb_val = cb_clf.predict_proba(X_val)[:, 1]
print(f'CatBoost xong ({time.time()-t0:.1f}s) -> ROC-AUC: {roc_auc_score(y_test, p_cb):.4f} | PR-AUC: {average_precision_score(y_test, p_cb):.4f}')

print('\n3/3. Training LightGBM...')
t0 = time.time()
lgb_clf = lgb.LGBMClassifier(
    n_estimators=600,
    num_leaves=31,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.5,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
p_lgb = lgb_clf.predict_proba(X_test)[:, 1]
p_lgb_val = lgb_clf.predict_proba(X_val)[:, 1]
print(f'LightGBM xong ({time.time()-t0:.1f}s) -> ROC-AUC: {roc_auc_score(y_test, p_lgb):.4f} | PR-AUC: {average_precision_score(y_test, p_lgb):.4f}')


1/3. Training Tuned XGBoost...
XGBoost xong (39.8s) -> ROC-AUC: 0.9722 | PR-AUC: 0.8464

2/3. Training CatBoost...
CatBoost xong (49.5s) -> ROC-AUC: 0.9627 | PR-AUC: 0.8300

3/3. Training LightGBM...
LightGBM xong (5.7s) -> ROC-AUC: 0.9546 | PR-AUC: 0.7773


In [4]:
# 3. Kết hợp xác suất (Soft Voting Ensemble)
p_ensemble_val = 0.5 * p_xgb_val + 0.3 * p_cb_val + 0.2 * p_lgb_val
p_ensemble = 0.5 * p_xgb + 0.3 * p_cb + 0.2 * p_lgb

print('==================================================')
print('KẾT QUẢ ĐA MÔ HÌNH ENSEMBLE TRÊN TẬP TEST:')
print(f'ROC-AUC Score:        {roc_auc_score(y_test, p_ensemble):.4f}')
print(f'PR-AUC Score (AUPRC): {average_precision_score(y_test, p_ensemble):.4f}')
print('==================================================')
summary(y_test, (p_ensemble >= 0.5).astype(int))


KẾT QUẢ ĐA MÔ HÌNH ENSEMBLE TRÊN TẬP TEST:
ROC-AUC Score:        0.9633
PR-AUC Score (AUPRC): 0.8366
accuracy_score: 0.9995
recall_score:    0.8108
precision_score: 0.8824
f1_score:        0.8451
COnfusion matrix:
 [[42640     8]
 [   14    60]]


In [5]:
# 4. Tối ưu hóa ngưỡng quyết định (Cost-Sensitive Threshold Tuning)
thresholds = np.linspace(0.1, 0.9, 81)
best_th, best_val_f1 = 0.5, 0.0
for th in thresholds:
    f1 = f1_score(y_val, (p_ensemble_val >= th).astype(int), zero_division=0)
    if f1 > best_val_f1:
        best_val_f1 = f1
        best_th = th

opt_threshold = float(best_th)
print(f'\nKẾT QUẢ TẠI NGƯỠNG TỐI ƯU T* = {opt_threshold:.2f}:')
print(f'ROC-AUC Score:        {roc_auc_score(y_test, p_ensemble):.4f}')
print(f'PR-AUC Score (AUPRC): {average_precision_score(y_test, p_ensemble):.4f}')
print('==================================================')
summary(y_test, (p_ensemble >= opt_threshold).astype(int))



KẾT QUẢ TẠI NGƯỠNG TỐI ƯU T* = 0.62:
ROC-AUC Score:        0.9633
PR-AUC Score (AUPRC): 0.8366
accuracy_score: 0.9996
recall_score:    0.7838
precision_score: 0.9206
f1_score:        0.8467
COnfusion matrix:
 [[42643     5]
 [   16    58]]


In [6]:
# 5. Vẽ Precision-Recall Curve & Chi phí theo ngưỡng
if 'thresholds' not in locals():
    thresholds = np.linspace(0.1, 0.9, 81)
if 'opt_threshold' not in locals():
    opt_threshold = 0.62

C_FN, C_FP = 200, 10
costs = []
for th in thresholds:
    pred_v = (p_ensemble_val >= th).astype(int)
    fn = ((y_val == 1) & (pred_v == 0)).sum()
    fp = ((y_val == 0) & (pred_v == 1)).sum()
    costs.append(fn * C_FN + fp * C_FP)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
prec, rec, _ = precision_recall_curve(y_test, p_ensemble)
ax1.plot(rec, prec, color='b', lw=2, label=f'PR-AUC = {average_precision_score(y_test, p_ensemble):.4f}')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curve (Ensemble)')
ax1.legend(loc='lower left')
ax1.grid(True)

ax2.plot(thresholds, costs, color='r', lw=2, label='Tổng chi phí tổn thất ($)')
ax2.axvline(opt_threshold, color='green', linestyle='--', label=f'Ngưỡng tối ưu T* = {opt_threshold:.2f}')
ax2.set_xlabel('Ngưỡng quyết định (Threshold)')
ax2.set_ylabel('Chi phí ($)')
ax2.set_title('Tổn thất kinh tế theo ngưỡng')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()
